<a href="https://colab.research.google.com/github/beingAnujChaudhary/Practical-Statistics-for-DS/blob/main/chapter_03_statistical_experiments_and_significance_testing/experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/Practical-Statistics-for-DS.git

# Move into repository
%cd /content/Practical-Statistics-for-DS

# Move into Chapter 3 folder
%cd chapter_03_statistical_experiments_and_significance_testing

Mounted at /content/drive
Cloning into 'Practical-Statistics-for-DS'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 168 (delta 102), reused 70 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 577.42 KiB | 12.29 MiB/s, done.
Resolving deltas: 100% (102/102), done.
/content/Practical-Statistics-for-DS
/content/Practical-Statistics-for-DS/chapter_03_statistical_experiments_and_significance_testing


# Chapter 03 Experiments: Statistical Experiments and Significance Testing

> Source: *Practical Statistics for Data Scientists, 2nd Edition*  
> Chapter 3: Statistical Experiments and Significance Testing

## Purpose

This notebook is for:
- Experimentation and intuition building
- Testing assumptions about statistical tests
- Understanding randomness and significance
- Exploring what happens when experiments go wrong
- Building reusable statistical utilities

Unlike `exercises.ipynb`, there are no fixed answers.

Goal:

> "How do I know whether a result is real or random?"

---

## 🧪 How to Use This Notebook

- **Experiments ≠ Exercises**: These are open-ended explorations designed to build your intuition for experimental design and statistical pitfalls.
- **Modify freely**: Change parameters, swap distributions, and observe how the statistical machinery holds up (or breaks).
- **Document insights**: Add your observations in markdown cells as you go.
- **Save interesting outputs**: Export plots to `output/` for your portfolio.

---

## Setup

<details>
<summary>Click to view solution</summary>

```python
# Imports
try:
    from utils.notebook_setup import *
except:
    import warnings
    warnings.filterwarnings("ignore")
    
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    from scipy import stats
    from statsmodels.stats.multitest import multipletests
    
    plt.style.use("seaborn-v0_8")
    np.random.seed(42)

# Create output directory
os.makedirs('output', exist_ok=True)

# Generic permutation test function (reusable utility)
def generic_permutation_test(group_a, group_b, stat_func, n_permutations=5000):
    """Permutation test for ANY statistic function."""
    obs_stat = stat_func(group_b) - stat_func(group_a)
    pooled = np.concatenate([group_a, group_b])
    n_a = len(group_a)
    
    perm_stats = np.zeros(n_permutations)
    for i in range(n_permutations):
        np.random.shuffle(pooled)
        perm_a = pooled[:n_a]
        perm_b = pooled[n_a:]
        perm_stats[i] = stat_func(perm_b) - stat_func(perm_a)
        
    p_value = np.mean(np.abs(perm_stats) >= np.abs(obs_stat))
    return obs_stat, p_value, perm_stats

print("Experiment notebook ready")
```
</details>



---

# Experiment 1: Can Randomness Fool Us?

## Question

Can two identical groups appear different?

### Hypothesis

Randomness may create fake differences.

<details>
<summary>Click to view solution</summary>

```python
group_a = np.random.normal(50, 10, 1000)
group_b = np.random.normal(50, 10, 1000)

plt.figure(figsize=(10, 6))
sns.histplot(group_a, label="Group A", kde=True, alpha=0.5)
sns.histplot(group_b, label="Group B", kde=True, alpha=0.5)
plt.legend()
plt.title("Two Random Groups (Same Distribution)")
plt.show()

t_stat, p_value = stats.ttest_ind(group_a, group_b)
print(f"P-value: {p_value:.4f}")
print("Both groups drawn from same distribution - any 'difference' is purely random.")
```
</details>

## Reflection

Questions:
1. Were groups actually different?
2. Did randomness still create variation?
3. Why do experiments need significance testing?

---

# Experiment 2: Effect Size vs P-value

## Question

How does effect size change significance?

### Hypothesis

Larger effect → smaller p-value.

<details>
<summary>Click to view solution</summary>

```python
effect_sizes = [50, 51, 53, 56]
results = []

for mean_value in effect_sizes:
    a = np.random.normal(50, 10, 1000)
    b = np.random.normal(mean_value, 10, 1000)
    _, p = stats.ttest_ind(a, b)
    results.append(p)

effect_df = pd.DataFrame({"Group B Mean": effect_sizes, "P-value": results})
print(effect_df)

plt.figure(figsize=(8, 5))
plt.plot(effect_sizes, results, marker="o", linewidth=2)
plt.axhline(0.05, linestyle="--", label="Threshold (α=0.05)")
plt.legend()
plt.xlabel("Effect Size (Group B Mean)")
plt.ylabel("P-value")
plt.title("Effect Size vs P-value")
plt.grid(alpha=0.3)
plt.show()
```
</details>

## Reflection

Questions:
1. What happened as effect increased?
2. Why do strong effects matter?
3. Can tiny effects still become significant?



---

# Experiment 3: Sample Size vs Significance

## Question

Can more data create significance even for tiny effects?

### Hypothesis

Larger datasets should detect smaller effects.

<details>
<summary>Click to view solution</summary>

```python
sample_sizes = [10, 50, 100, 500, 5000]
p_values = []

for n in sample_sizes:
    a = np.random.normal(50, 10, n)
    b = np.random.normal(51, 10, n)  # Tiny 1-point difference
    _, p = stats.ttest_ind(a, b)
    p_values.append(p)

plt.figure(figsize=(8, 5))
plt.plot(sample_sizes, p_values, marker="o", linewidth=2)
plt.axhline(0.05, linestyle="--", label="Threshold (α=0.05)")
plt.xscale("log")
plt.legend()
plt.xlabel("Sample Size (log scale)")
plt.ylabel("P-value")
plt.title("Sample Size vs Significance\n(Tiny 1-point effect)")
plt.grid(alpha=0.3)
plt.show()

for n, p in zip(sample_sizes, p_values):
    sig = "Significant" if p < 0.05 else "Not Significant"
    print(f"n={n:>5}: p={p:.4f} → {sig}")
```
</details>

## Reflection

Questions:
1. What happened as sample size increased?
2. Why can tiny effects become significant?
3. Why is statistical significance sometimes misleading?



---

# Experiment 4: False Positives Simulation

## Question

How often do false positives happen when there's no real effect?

### Hypothesis

Around 5% for α = 0.05.

<details>
<summary>Click to view solution</summary>

```python
false_positive_count = 0

for _ in range(1000):
    a = np.random.normal(50, 10, 100)
    b = np.random.normal(50, 10, 100)  # Same distribution - no real difference
    _, p = stats.ttest_ind(a, b)
    if p < 0.05:
        false_positive_count += 1

rate = false_positive_count / 1000
print(f"False Positive Rate: {rate:.3f}")
print(f"Expected: ~0.050 (5%)")
print(f"Number of false positives in 1000 tests: {false_positive_count}")
```
</details>

## Reflection

Questions:
1. Was rate close to 5%?
2. Why do false positives happen?
3. Why is experimentation risky without proper controls?


---

# Experiment 5: Type II Error Investigation

## Question

Can we miss a real effect with small samples?

<details>
<summary>Click to view solution</summary>

```python
# Simulate multiple small-sample experiments with a REAL effect
missed_effects = 0
n_trials = 100

for _ in range(n_trials):
    small_a = np.random.normal(50, 10, 10)
    small_b = np.random.normal(53, 10, 10)  # Real 3-point difference
    _, p = stats.ttest_ind(small_a, small_b)
    if p >= 0.05:
        missed_effects += 1

print(f"Real effect missed in {missed_effects}/{n_trials} trials")
print(f"Type II Error Rate: {missed_effects/n_trials:.1%}")
print("With n=10, even a real 3-point difference is often not detected.")
```
</details>

## Reflection

Questions:
1. Did significance fail to detect the real effect?
2. Could effect still exist despite non-significant result?
3. Why are small samples risky for decision-making?


---

# Experiment 6: P-hacking Simulation

## Question

What happens if we keep testing on random data?

### Hypothesis

Eventually, small p-values appear by chance.

<details>
<summary>Click to view solution</summary>

```python
p_values = []

for _ in range(100):
    a = np.random.normal(50, 10, 100)
    b = np.random.normal(50, 10, 100)  # No real difference
    _, p = stats.ttest_ind(a, b)
    p_values.append(p)

print(f"Smallest P-value from 100 tests: {min(p_values):.4f}")
print(f"Number of 'significant' results (p<0.05): {sum(np.array(p_values) < 0.05)}")
print(f"Number of 'significant' results (p<0.01): {sum(np.array(p_values) < 0.01)}")
print("\nThis demonstrates how easy it is to find 'significant' results by running many tests.")
```
</details>

## Reflection

Questions:
1. Did tiny p-values appear despite no real effect?
2. Could this fool analysts into false discoveries?
3. Why is p-hacking dangerous in data science?


---

# Experiment 7: Permutation Testing Intuition

## Question

Can shuffling labels help us understand if observed differences are real?

<details>
<summary>Click to view solution</summary>

```python
# Create A/B test data with a small real effect
group_a = np.random.binomial(1, 0.12, 1000)
group_b = np.random.binomial(1, 0.14, 1000)

observed_difference = group_b.mean() - group_a.mean()
print(f"Observed Difference: {observed_difference:.4f}")

# Permutation test
combined = np.concatenate([group_a, group_b])
permutation_results = []

for _ in range(1000):
    shuffled = np.random.permutation(combined)
    new_a = shuffled[:1000]
    new_b = shuffled[1000:]
    difference = new_b.mean() - new_a.mean()
    permutation_results.append(difference)

# Visualize
plt.figure(figsize=(10, 6))
sns.histplot(permutation_results, bins=30, kde=True)
plt.axvline(observed_difference, linestyle="--", color='red', linewidth=2, label="Observed Difference")
plt.axvline(-observed_difference, linestyle="--", color='red', linewidth=2, alpha=0.5)
plt.legend()
plt.title("Permutation Test: Null Distribution")
plt.xlabel("Difference in Means")
plt.show()

p_value_perm = np.mean(np.abs(permutation_results) >= np.abs(observed_difference))
print(f"Permutation P-value: {p_value_perm:.4f}")
```
</details>

## Reflection

Questions:
1. Was observed result unusual compared to shuffled distribution?
2. Why does shuffling matter for inference?
3. Why are permutation tests intuitive and assumption-free?


---

# Experiment 8: Beyond the Mean — Permutation Tests for Any Metric

### Goal

The book demonstrates permutation tests using the difference in means. But in Data Science, we often care about other metrics: the median (for skewed revenue data), the variance (for model latency stability), or the 90th percentile (for SLA compliance).

Because permutation tests are simulation-based, we can swap the test statistic without needing to derive new mathematical formulas.

<details>
<summary>Click to view solution</summary>

```python
# Simulate web session times (highly skewed, typical of web data)
np.random.seed(42)
page_a = np.random.exponential(scale=120, size=200)
page_b = np.random.exponential(scale=135, size=200)  # Slightly better, but high variance

# Test 1: Difference in Medians (Robust to outliers)
obs_median, p_median, _ = generic_permutation_test(page_a, page_b, np.median)
print(f"Median Diff: {obs_median:.2f}s | p-value: {p_median:.4f}")

# Test 2: Difference in 90th Percentile (SLA focus)
p90_func = lambda x: np.percentile(x, 90)
obs_p90, p_p90, _ = generic_permutation_test(page_a, page_b, p90_func)
print(f"90th Pctl Diff: {obs_p90:.2f}s | p-value: {p_p90:.4f}")

# Test 3: Difference in Variance (Consistency focus)
obs_var, p_var, _ = generic_permutation_test(page_a, page_b, np.var)
print(f"Variance Diff: {obs_var:.2f} | p-value: {p_var:.4f}")
```
</details>

### 🤔 Reflection Prompts
1. Notice how the p-values differ depending on the metric. Why might the mean be misleading for web session data?
2. **ML Relevance**: If you are comparing the inference latency of Model A vs. Model B, why is testing the 99th percentile (P99) via a permutation test superior to a standard t-test on the mean?
3. Try changing the sample size to n=20. How does the p-value for the variance test react compared to the median test?



---

# Experiment 9: Simulating P-Hacking and the "Vast Search Effect"

### Goal

The book warns about the "vast search effect" (data snooping). If you test enough hypotheses on random noise, you will find a statistically significant result purely by chance. This experiment simulates a common ML trap: Feature Selection on pure noise.

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)

# Simulate a target variable and 100 predictor features, ALL pure random noise
n_samples = 500
y = np.random.normal(0, 1, n_samples)
X = np.random.normal(0, 1, (n_samples, 100))

# Calculate Pearson correlation and p-values for all 100 features
p_values_feat = []
for i in range(100):
    _, p_val = stats.pearsonr(X[:, i], y)
    p_values_feat.append(p_val)

p_values_feat = np.array(p_values_feat)

# 1. Uncorrected (The P-Hacking approach)
sig_uncorrected = np.sum(p_values_feat < 0.05)
print(f"Uncorrected: Found {sig_uncorrected} 'significant' features out of 100 (Expected ~5)")

# 2. Bonferroni Correction (Strict)
reject_bonf, pvals_bonf, _, _ = multipletests(p_values_feat, alpha=0.05, method='bonferroni')
print(f"Bonferroni: Found {np.sum(reject_bonf)} significant features.")

# 3. Benjamini-Hochberg FDR (Standard in DS/Genomics)
reject_fdr, pvals_fdr, _, _ = multipletests(p_values_feat, alpha=0.05, method='fdr_bh')
print(f"FDR (BH): Found {np.sum(reject_fdr)} significant features.")
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Visualize the P-Value Distribution
plt.figure(figsize=(8, 5))
plt.hist(p_values_feat, bins=20, color='skyblue', edgecolor='black', alpha=0.8)
plt.axvline(0.05, color='red', linestyle='--', linewidth=2, label='Alpha = 0.05')
plt.xlabel('P-value')
plt.ylabel('Frequency')
plt.title('Distribution of P-Values for 100 Pure Noise Features')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('output/p_hacking_distribution.png', dpi=300)
plt.show()
```
</details>

### 🤔 Reflection Prompts
1. The histogram of p-values for pure noise should be roughly uniform (flat). What does it mean if you see a spike near 0.0 in a real dataset?
2. **ML Relevance**: If you use `SelectKBest` or filter features by p-value before training a model, how does the Vast Search Effect lead to data leakage and overfitting?
3. Why is FDR (Benjamini-Hochberg) generally preferred over Bonferroni in high-dimensional data science tasks?



---

# Experiment 10: Bandit Algorithms — Epsilon-Greedy vs. Thompson Sampling

### Goal

The book introduces the Epsilon-Greedy Multi-Arm Bandit. However, in modern industry (ad-tech, recommendation systems), Thompson Sampling is often preferred because it naturally balances exploration and exploitation using Bayesian updates. Let's build and compare them.

<details>
<summary>Click to view solution</summary>

```python
class EpsilonGreedyBandit:
    def __init__(self, n_arms, epsilon=0.1):
        self.epsilon = epsilon
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)
        
    def select_arm(self):
        if np.random.random() < self.epsilon:
            return np.random.randint(len(self.values))
        return np.argmax(self.values)
        
    def update(self, arm, reward):
        self.counts[arm] += 1
        n = self.counts[arm]
        self.values[arm] += (reward - self.values[arm]) / n

class ThompsonSamplingBandit:
    def __init__(self, n_arms):
        self.alpha = np.ones(n_arms)
        self.beta = np.ones(n_arms)
        
    def select_arm(self):
        samples = [np.random.beta(self.alpha[i], self.beta[i]) for i in range(len(self.alpha))]
        return np.argmax(samples)
        
    def update(self, arm, reward):
        if reward == 1:
            self.alpha[arm] += 1
        else:
            self.beta[arm] += 1

# Simulation Parameters
true_rates = [0.02, 0.05, 0.08]  # Arm 2 is the true winner
n_steps = 5000

def run_simulation(bandit, true_rates, n_steps):
    rewards = []
    regret = []
    optimal_arm = np.argmax(true_rates)
    
    for _ in range(n_steps):
        arm = bandit.select_arm()
        reward = np.random.binomial(1, true_rates[arm])
        bandit.update(arm, reward)
        rewards.append(reward)
        regret.append(true_rates[optimal_arm] - true_rates[arm])
        
    return np.cumsum(rewards), np.cumsum(regret)

# Run both
np.random.seed(42)
eg_bandit = EpsilonGreedyBandit(n_arms=3, epsilon=0.1)
ts_bandit = ThompsonSamplingBandit(n_arms=3)

eg_cum_reward, eg_cum_regret = run_simulation(eg_bandit, true_rates, n_steps)
ts_cum_reward, ts_cum_regret = run_simulation(ts_bandit, true_rates, n_steps)
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Visualize Cumulative Regret
plt.figure(figsize=(10, 6))
plt.plot(eg_cum_regret, label='Epsilon-Greedy (ε=0.1)', linewidth=2, color='salmon')
plt.plot(ts_cum_regret, label='Thompson Sampling', linewidth=2, color='teal')
plt.xlabel('Steps (Users)')
plt.ylabel('Cumulative Regret')
plt.title('Bandit Performance: Cumulative Regret over Time')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('output/bandit_regret_comparison.png', dpi=300)
plt.show()

print("Note: Lower cumulative regret = faster convergence to optimal arm.")
print("Thompson Sampling typically converges faster than Epsilon-Greedy.")
```
</details>

### 🤔 Reflection Prompts
1. Notice how Thompson Sampling converges to the optimal arm faster (lower regret slope) than Epsilon-Greedy. Why?
2. **ML Relevance**: How could you adapt Thompson Sampling to hyperparameter tuning (e.g., selecting the best learning rate during model training)?
3. What happens to Thompson Sampling if the reward is continuous (e.g., revenue) instead of binary (click/no-click)?



---

# Experiment 11: The Reality of Statistical Power

### Goal

Business stakeholders often want to run A/B tests for "just one week" regardless of the baseline conversion rate. This experiment visualizes Statistical Power to show why small effect sizes require massive sample sizes, and why most underpowered tests are a waste of engineering time.

<details>
<summary>Click to view solution</summary>

```python
def simulate_power(n_per_group, p_control, p_treatment, alpha=0.05, n_simulations=500):
    """Estimate power via Monte Carlo simulation."""
    significant_count = 0
    for _ in range(n_simulations):
        conv_a = np.random.binomial(1, p_control, n_per_group)
        conv_b = np.random.binomial(1, p_treatment, n_per_group)
        
        table = np.array([
            [sum(conv_a), n_per_group - sum(conv_a)],
            [sum(conv_b), n_per_group - sum(conv_b)]
        ])
        _, p_val, _, _ = stats.chi2_contingency(table)
        if p_val < alpha:
            significant_count += 1
            
    return significant_count / n_simulations

# Define scenarios
sample_sizes_power = np.linspace(500, 20000, 15).astype(int)
baseline_rate = 0.05  # 5% baseline conversion

# Scenario A: 10% Relative Lift (0.050 -> 0.055)
# Scenario B: 50% Relative Lift (0.050 -> 0.075)
powers_small_lift = []
powers_large_lift = []

print("Simulating Power... (This may take a moment)")
for n in sample_sizes_power:
    p_small = simulate_power(n, baseline_rate, baseline_rate * 1.10, n_simulations=300)
    p_large = simulate_power(n, baseline_rate, baseline_rate * 1.50, n_simulations=300)
    powers_small_lift.append(p_small)
    powers_large_lift.append(p_large)
    if n % 5000 == 0:
        print(f"  n={n}: Small lift power={p_small:.1%}, Large lift power={p_large:.1%}")
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Visualize Power Curves
plt.figure(figsize=(10, 6))
plt.plot(sample_sizes_power, powers_small_lift, 'o-',
         label='10% Relative Lift (0.050 → 0.055)', color='purple', linewidth=2)
plt.plot(sample_sizes_power, powers_large_lift, 's-',
         label='50% Relative Lift (0.050 → 0.075)', color='green', linewidth=2)
plt.axhline(0.80, color='red', linestyle='--', label='80% Power Target')
plt.xlabel('Sample Size per Group')
plt.ylabel('Statistical Power')
plt.title('Power Curves: Detecting Different Effect Sizes')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('output/power_curves.png', dpi=300)
plt.show()
```
</details>

### 🤔 Reflection Prompts
1. Look at the sample size required to achieve 80% power for a 10% relative lift. Is this feasible for your company's daily traffic?
2. **ML Relevance**: When evaluating an offline ML model (e.g., comparing AUC of Model A vs Model B on a holdout set), how does the concept of "Power" dictate the size of your test set?
3. What is the danger of "peeking" at an A/B test every day and stopping it the moment p < 0.05?



---

# Experiment 12: Outlier Impact on Statistical Tests

### Goal

Does the presence of an outlier in a small sample completely break the classical t-test, while the permutation test remains valid?

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)

# Generate small samples from the same distribution (Null is TRUE)
group_a = np.random.normal(100, 15, 15)
group_b = np.random.normal(100, 15, 15)

# Inject one massive outlier into Group B (e.g., data entry error)
group_b[0] = 500

print("With outlier injected into Group B:\n")

# Classical T-Test
t_stat, t_pval = stats.ttest_ind(group_a, group_b, equal_var=False)
print(f"Classical Welch's T-Test p-value: {t_pval:.4f}")

# Permutation Test (using the mean)
_, perm_pval_mean, _ = generic_permutation_test(group_a, group_b, np.mean, n_permutations=3000)
print(f"Permutation Test (Mean) p-value: {perm_pval_mean:.4f}")

# Permutation Test (using the median - robust!)
_, perm_pval_median, _ = generic_permutation_test(group_a, group_b, np.median, n_permutations=3000)
print(f"Permutation Test (Median) p-value: {perm_pval_median:.4f}")

print("\nInsight: The outlier caused issues in mean-based tests, but the median-based permutation test is robust.")
```
</details>



---

# ML Relevance

## Why Chapter 3 matters in ML

Machine learning improvements can be fake improvements. Questions ML engineers ask:
- Did model actually improve?
- Was it randomness?
- Is the improvement practically meaningful?

Important concepts:
- A/B testing for model comparison
- False positives in feature selection
- Overfitting detection
- Reliable evaluation
- Significance testing
- Uncertainty quantification



## 📝 Experiment Log

| Date | Experiment | Key Insight | ML Relevance |
|------|-----------|-------------|--------------|
| | Permutation beyond mean | Can test P99 latency without math formulas | Crucial for SLA monitoring and model serving |
| | P-Hacking Simulation | Pure noise yields 5% false positives | Feature selection must use FDR or holdout sets |
| | Thompson Sampling | Bayesian bandits converge faster than Epsilon-Greedy | Better for real-time recommendation systems |
| | Power Curves | Small effects need massive samples | Dictates test set sizes for model evaluation |

*Update this table as you complete experiments.*

---

## 📊 Exporting Your Findings

<details>
<summary>Click to view solution</summary>

```python
# Example: Save the power curves plot
plt.figure(figsize=(10, 6))
plt.plot(sample_sizes_power, powers_small_lift, 'o-',
         label='10% Relative Lift (0.050 → 0.055)', color='purple', linewidth=2)
plt.plot(sample_sizes_power, powers_large_lift, 's-',
         label='50% Relative Lift (0.050 → 0.075)', color='green', linewidth=2)
plt.axhline(0.80, color='red', linestyle='--', label='80% Power Target')
plt.xlabel('Sample Size per Group')
plt.ylabel('Statistical Power')
plt.title('Power Curves: Detecting Different Effect Sizes', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('output/power_curves_final.png', dpi=300, bbox_inches='tight')
print("Saved plot to output/power_curves_final.png")
plt.show()
```
</details>

---

# Challenge Experiment

Create experiments for:
1. Tiny effect + huge sample — does significance always appear?
2. Huge effect + tiny sample — can we still detect it?
3. False positive simulation with varying alpha levels
4. P-hacking experiment with different numbers of tests
5. Permutation vs t-test comparison under different conditions
6. Bootstrap A/B test vs Permutation test comparison

---

# Progress Checklist

- [ ] Tested randomness with identical groups
- [ ] Compared effect sizes and their impact on p-values
- [ ] Explored sample size vs significance relationship
- [ ] Simulated false positives (Type I error)
- [ ] Investigated Type II error with small samples
- [ ] Tested p-hacking with multiple comparisons
- [ ] Practised permutation testing
- [ ] Built generic permutation test for any metric
- [ ] Compared median, P90, and variance permutation tests
- [ ] Simulated vast search effect with 100 noise features
- [ ] Applied Bonferroni and FDR corrections
- [ ] Implemented Epsilon-Greedy and Thompson Sampling bandits
- [ ] Compared bandit regret curves
- [ ] Simulated power curves for different effect sizes
- [ ] Tested outlier impact on classical vs permutation tests
- [ ] Connected ideas to ML
- [ ] Exported key plots to output/

---

## ✅ Next Steps

1. **Pick 2-3 experiments** to run deeply this week.
2. **Modify parameters** (e.g., change the baseline conversion rate in the Power experiment to match your actual business metrics).
3. **Export** your favorite plot to `output/` for your portfolio.

---

> **Pro Tip**: The `generic_permutation_test` function you built in this notebook is a highly valuable utility. Save it to your `utils/stats_helpers.py` file. In the real world, business stakeholders rarely care about the "mean"—they care about the median revenue or the 95th percentile of load times. Being able to run a rigorous significance test on any metric is a superpower in Data Science.

*Happy experimenting! 🧪✨*